In [ ]:
# =========================================
# 1. Imports
# =========================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from pathlib import Path

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 120)


In [ ]:
# =========================================
# 2. Load dataset
# =========================================
DATA_PATH = Path("../data/raw/airlines_delay.csv")  # <- ggf. Dateinamen anpassen

df = pd.read_csv(DATA_PATH)

print("Shape:", df.shape)
df.head()


In [ ]:
# =========================================
# 3. Basic overview
# =========================================
print("First 5 rows:")
display(df.head())

print("\nColumn names:")
print(df.columns.tolist())

print("\nData types:")
display(df.dtypes)

print("\nMissing values:")
display(df.isna().sum())

print("\nDuplicated rows:", df.duplicated().sum())


In [ ]:
# =========================================
# 4. Basic statistics
# =========================================
display(df.describe(include="all"))


In [ ]:
# =========================================
# 5. Target variable distribution
# =========================================
target_counts = df["Delay"].value_counts().sort_index()
target_ratio = df["Delay"].value_counts(normalize=True).sort_index() * 100

print("Target counts:")
display(target_counts)

print("Target percentage:")
display(target_ratio.round(2))

plt.figure(figsize=(6, 4))
target_counts.plot(kind="bar")
plt.title("Target Distribution: Delay")
plt.xlabel("Delay (0 = No, 1 = Yes)")
plt.ylabel("Count")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()


In [ ]:
# =========================================
# 6. Unique values per column
# =========================================
unique_values = pd.DataFrame({
    "column": df.columns,
    "n_unique": [df[col].nunique() for col in df.columns]
}).sort_values(by="n_unique", ascending=False)

display(unique_values)


In [ ]:
# =========================================
# 7. Feature engineering for EDA
# =========================================
# Time is in HHMM format, e.g. 1330 -> 13:30
df_eda = df.copy()

df_eda["DepartureHour"] = df_eda["Time"] // 100
df_eda["DepartureMinute"] = df_eda["Time"] % 100
df_eda["IsWeekend"] = df_eda["DayOfWeek"].isin([6, 7]).astype(int)

print(df_eda[["Time", "DepartureHour", "DepartureMinute", "IsWeekend"]].head())


In [ ]:
# =========================================
# 8. Sanity checks for engineered columns
# =========================================
print("DepartureHour min/max:", df_eda["DepartureHour"].min(), df_eda["DepartureHour"].max())
print("DepartureMinute min/max:", df_eda["DepartureMinute"].min(), df_eda["DepartureMinute"].max())

invalid_hours = df_eda[(df_eda["DepartureHour"] < 0) | (df_eda["DepartureHour"] > 23)]
invalid_minutes = df_eda[(df_eda["DepartureMinute"] < 0) | (df_eda["DepartureMinute"] > 59)]

print("Invalid hour rows:", len(invalid_hours))
print("Invalid minute rows:", len(invalid_minutes))


In [ ]:
# =========================================
# 9. Delay rate by airline
# =========================================
airline_delay = (
    df_eda.groupby("Airline")["Delay"]
    .agg(["count", "mean"])
    .sort_values("mean", ascending=False)
)

airline_delay["delay_rate_percent"] = airline_delay["mean"] * 100
display(airline_delay.head(15))

plt.figure(figsize=(10, 5))
airline_delay["delay_rate_percent"].plot(kind="bar")
plt.title("Delay Rate by Airline")
plt.xlabel("Airline")
plt.ylabel("Delay Rate (%)")
plt.tight_layout()
plt.show()


In [ ]:
# =========================================
# 10. Delay rate by origin airport
# =========================================
origin_delay = (
    df_eda.groupby("AirportFrom")["Delay"]
    .agg(["count", "mean"])
    .sort_values(["count", "mean"], ascending=[False, False])
)

# only airports with enough observations
origin_delay_filtered = origin_delay[origin_delay["count"] >= 100]
origin_delay_filtered = origin_delay_filtered.sort_values("mean", ascending=False).head(20)
origin_delay_filtered["delay_rate_percent"] = origin_delay_filtered["mean"] * 100

display(origin_delay_filtered)

plt.figure(figsize=(12, 5))
origin_delay_filtered["delay_rate_percent"].plot(kind="bar")
plt.title("Top 20 Origin Airports by Delay Rate (min 100 flights)")
plt.xlabel("Origin Airport")
plt.ylabel("Delay Rate (%)")
plt.tight_layout()
plt.show()


In [ ]:
# =========================================
# 11. Delay rate by destination airport
# =========================================
dest_delay = (
    df_eda.groupby("AirportTo")["Delay"]
    .agg(["count", "mean"])
    .sort_values(["count", "mean"], ascending=[False, False])
)

dest_delay_filtered = dest_delay[dest_delay["count"] >= 100]
dest_delay_filtered = dest_delay_filtered.sort_values("mean", ascending=False).head(20)
dest_delay_filtered["delay_rate_percent"] = dest_delay_filtered["mean"] * 100

display(dest_delay_filtered)

plt.figure(figsize=(12, 5))
dest_delay_filtered["delay_rate_percent"].plot(kind="bar")
plt.title("Top 20 Destination Airports by Delay Rate (min 100 flights)")
plt.xlabel("Destination Airport")
plt.ylabel("Delay Rate (%)")
plt.tight_layout()
plt.show()


In [ ]:
# =========================================
# 12. Delay rate by day of week
# =========================================
day_delay = df_eda.groupby("DayOfWeek")["Delay"].mean() * 100
display(day_delay)

plt.figure(figsize=(7, 4))
day_delay.plot(kind="bar")
plt.title("Delay Rate by Day of Week")
plt.xlabel("DayOfWeek")
plt.ylabel("Delay Rate (%)")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()


In [ ]:
# =========================================
# 13. Delay rate by departure hour
# =========================================
hour_delay = df_eda.groupby("DepartureHour")["Delay"].mean() * 100
display(hour_delay)

plt.figure(figsize=(10, 4))
hour_delay.plot(kind="bar")
plt.title("Delay Rate by Departure Hour")
plt.xlabel("Departure Hour")
plt.ylabel("Delay Rate (%)")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()


In [ ]:
# =========================================
# 14. Flight length distribution
# =========================================
plt.figure(figsize=(8, 4))
plt.hist(df_eda["Length"], bins=40)
plt.title("Distribution of Flight Length")
plt.xlabel("Length")
plt.ylabel("Frequency")
plt.tight_layout()
plt.show()

print("Length summary:")
display(df_eda["Length"].describe())


In [ ]:
# =========================================
# 15. Delay rate by flight length bins
# =========================================
df_eda["LengthBin"] = pd.cut(
    df_eda["Length"],
    bins=[0, 60, 120, 180, 240, 360, 600],
    include_lowest=True
)

length_delay = df_eda.groupby("LengthBin", observed=False)["Delay"].mean() * 100
display(length_delay)

plt.figure(figsize=(9, 4))
length_delay.plot(kind="bar")
plt.title("Delay Rate by Flight Length Bin")
plt.xlabel("Flight Length Bin")
plt.ylabel("Delay Rate (%)")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


In [ ]:
# =========================================
# 16. Top routes by number of flights
# =========================================
df_eda["Route"] = df_eda["AirportFrom"] + "-" + df_eda["AirportTo"]

top_routes = df_eda["Route"].value_counts().head(20)
display(top_routes)

plt.figure(figsize=(12, 5))
top_routes.plot(kind="bar")
plt.title("Top 20 Routes by Number of Flights")
plt.xlabel("Route")
plt.ylabel("Number of Flights")
plt.tight_layout()
plt.show()


In [ ]:
# =========================================
# 17. Delay rate by route (filtered)
# =========================================
route_delay = (
    df_eda.groupby("Route")["Delay"]
    .agg(["count", "mean"])
)

route_delay_filtered = route_delay[route_delay["count"] >= 100]
route_delay_filtered = route_delay_filtered.sort_values("mean", ascending=False).head(20)
route_delay_filtered["delay_rate_percent"] = route_delay_filtered["mean"] * 100

display(route_delay_filtered)

plt.figure(figsize=(12, 5))
route_delay_filtered["delay_rate_percent"].plot(kind="bar")
plt.title("Top 20 Routes by Delay Rate (min 100 flights)")
plt.xlabel("Route")
plt.ylabel("Delay Rate (%)")
plt.tight_layout()
plt.show()


In [ ]:
# =========================================
# 18. Check whether Flight column looks useful
# =========================================
print("Unique flight numbers:", df_eda["Flight"].nunique())
print("Total rows:", len(df_eda))

flight_delay = (
    df_eda.groupby("Flight")["Delay"]
    .agg(["count", "mean"])
    .sort_values("count", ascending=False)
)

display(flight_delay.head(20))


In [ ]:
# =========================================
# 19. Correlation for numeric columns
# =========================================
numeric_cols = ["Flight", "DayOfWeek", "Time", "Length", "DepartureHour", "DepartureMinute", "Delay"]

corr = df_eda[numeric_cols].corr(numeric_only=True)
display(corr)

plt.figure(figsize=(8, 6))
plt.imshow(corr, interpolation="nearest")
plt.title("Correlation Matrix (Numeric Features)")
plt.colorbar()
plt.xticks(range(len(corr.columns)), corr.columns, rotation=45)
plt.yticks(range(len(corr.columns)), corr.columns)
plt.tight_layout()
plt.show()


In [ ]:
# =========================================
# 20. Recommended modeling dataset preview
# =========================================
# Based on current understanding:
# - drop id (identifier only)
# - likely drop Flight (high-cardinality / weak generalization)
# - convert Time -> DepartureHour
# - keep Delay as target

model_df = df_eda.copy()

model_df = model_df.drop(columns=["id", "Flight", "Time", "DepartureMinute"], errors="ignore")

print("Modeling dataset shape:", model_df.shape)
display(model_df.head())
print(model_df.dtypes)


In [ ]:
# =========================================
# 21. Save processed EDA version (optional)
# =========================================
OUTPUT_PATH = Path("../data/processed/airlines_delay_eda.csv")
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)

model_df.to_csv(OUTPUT_PATH, index=False)

print(f"Saved processed EDA dataset to: {OUTPUT_PATH}")


In [ ]:
# =========================================
# 22. Final EDA summary for presentation
# =========================================
print("EDA SUMMARY")
print("-----------")
print(f"Rows: {df.shape[0]}")
print(f"Columns: {df.shape[1]}")
print(f"Missing values total: {df.isna().sum().sum()}")
print(f"Duplicate rows: {df.duplicated().sum()}")

print("\nTarget distribution (%):")
print((df['Delay'].value_counts(normalize=True) * 100).round(2).sort_index())

print("\nSuggested modeling decisions:")
print("- Use Delay as target")
print("- Drop id")
print("- Likely drop Flight")
print("- Transform Time into DepartureHour")
print("- Keep Airline, AirportFrom, AirportTo, DayOfWeek, Length")
